# LELA60341: Research Methods in Computational and Corpus Linguistics Week 8

## A logistic regression model, built from parts

In the LELA60331 seminars you have built **logistic regression** step by step, as a long series of notebook cells, and in Week 7 you trained it on simulated data. In this workbook you will build the same classifier out of **classes**, using what you learned about classes in Week 7. Rather than one big class, the model is put together from a few small parts, each with one job:

| Part of the model | Its job |
|---|---|
| `Parameter` | holds numbers that training changes (weights or biases), with their gradients |
| `Module` | the parent class of every layer and every model |
| `Linear` | a **layer**: turns each example's features into scores, and works out the gradients for its weights |
| `BCEWithLogitsLoss` | measures how wrong the scores are, and where the gradients start |
| `SGD` | the **optimiser**: updates the parameters by gradient descent |
| `LogisticRegression` | the **model**: made from a `Linear` layer |

By the end of this workbook you will train the model with a loop like this one, in which each line does one job:

```python
for epoch in range(n_epochs):
    optimizer.zero_grad()                 # forget the old gradients
    scores = model(X)                     # the forward pass
    loss = loss_fn(scores, y)             # how wrong is it?
    model.backward(loss_fn.grad())        # the backward pass: the gradients
    optimizer.step()                      # update the parameters
```

| Part | Topic |
|---|---|
| 1 | From a script to classes |
| 2 | The data as a matrix |
| 3 | Parameters and modules |
| 4 | A linear layer |
| 5 | A loss and an optimiser |
| 6 | The model and the training loop |
| 7 | Using and evaluating the model |

Read each explanation, run the example cells, then try the 🏋️ exercises. In an exercise, replace each `???` with your own code, then run the ✅ check cell underneath. When you see a 🔮 **Predict** prompt, decide what you think will happen *before* you run the cell. Parts marked ⭐ **Challenge** are optional.

In [ ]:
# ▶ Run this cell first. It defines the check() helper used by every exercise.
def check(label, condition):
    """Print a tick or a cross instead of crashing, so you can keep working."""
    try:
        passed = bool(condition() if callable(condition) else condition)
    except Exception as err:
        passed = False
        label += f"  (error: {type(err).__name__}: {err})"
    print(("✅ " if passed else "❌ ") + label)
    return passed

---
# Part 1 · From a script to classes

In the LELA60331 seminars, each step of the classifier is a separate cell, and the cells share their results through **global names**. Even for the simulated data in LELA60331 Week 7 there are a lot of them:

| Name | What it holds |
|---|---|
| `x`, `y`, `X` | the features and the labels |
| `weights`, `bias` | the model's parameters |
| `lr`, `n_epochs` | the training settings |
| `q`, `errors`, `loss_history` | the probabilities, the errors and the losses |
| `score`, `sigmoid`, `average_loss`, `mat_vec`, ... | the functions that work on them |

This works, but it is fragile:

* Run the cells in the wrong order, or re-run one by mistake, and the names hold the wrong values. (In LELA60331 Week 7, `q` holds the probabilities from whichever cell you ran last: the untrained model or the trained one.)
* You can't easily have **two** models at once, because they would both want the name `weights`.
* To use the classifier on new data, or to build a bigger model, you have to remember which cells to copy, and in which order.

Classes solve these problems. Each class bundles some **data** (as instance variables) with the **operations** on it (as methods), as in Week 7. Here we go one step further, and split the classifier into several small classes, each with one job (see the table at the top). Small parts are easier to write, to test and to reuse: the same `Linear` layer, loss and optimiser can be put together in different ways to make different models.

### 🏋️ Exercise 1 · Practice — Which part does what?

For each job below, say which part of the model should do it: `"Parameter"`, `"Linear"`, `"BCEWithLogitsLoss"` or `"SGD"`. (Look at the table at the top.)

In [ ]:
jobs = {
    "hold the weights and their gradients": ???,
    "turn the features of each example into a score": ???,
    "measure how wrong the scores are": ???,
    "update the weights by gradient descent": ???,
}

In [ ]:
# ✅ Check
check("Parameter", lambda: jobs["hold the weights and their gradients"] == "Parameter")
check("Linear", lambda: jobs["turn the features of each example into a score"] == "Linear")
check("the loss", lambda: jobs["measure how wrong the scores are"] == "BCEWithLogitsLoss")
check("the optimiser", lambda: jobs["update the weights by gradient descent"] == "SGD")

---
# Part 2 · The data as a matrix

Here is the simulated data from LELA60331 Weeks 5 and 7: 50 texts, each with two features (the log counts of positive and of negative words) and a label (1 = positive, 0 = negative). As in the seminar, `x[0]` holds feature 1 for every text and `x[1]` holds feature 2. Run the cell.

In [ ]:
import math
import random
import matplotlib.pyplot as plt

random.seed(10)
x = [[], []]
y = []
for i in range(50):
    if random.random() > 0.5:          # a negative text: fewer positive than negative words
        x[0].append(random.gauss(2, 1))
        x[1].append(random.gauss(3, 1))
        y.append(0)
    else:                              # a positive text: more positive than negative words
        x[0].append(random.gauss(3, 1))
        x[1].append(random.gauss(2, 1))
        y.append(1)
print(len(y), "texts,", sum(y), "of them positive")

In LELA60331 Week 7, Part 4 you saw that it is more convenient to store the data as a **matrix** `X` with **one row per text** and one column per feature, so that one matrix-vector multiplication gives the score of every text. You also saw that the gradients need the **transpose** of `X`: the same numbers with the rows turned into columns, one row per feature.

All our layers and models will take their data in the form `X`: a list of examples, one row each. A list of examples processed together is called a **batch**.

### 🏋️ Exercise 2 · Practice — Rows and the transpose

1. `X`: a list with one row `[x1, x2]` for each text (as in LELA60331 Week 7, Activities 6 and 7)
2. Complete `transpose(matrix)`, which returns a new matrix whose row `j` is column `j` of `matrix`. *Hint:* column `j` is `[row[j] for row in matrix]` (Week 4, Part 3), and there is one column for each element of the first row.

In [ ]:
X = ???                  # 1.

def transpose(matrix):
    return ???   # 2.

print(len(X), len(X[0]), X[0])
print(transpose([[1, 2, 3], [4, 5, 6]]))       # you should get [[1, 4], [2, 5], [3, 6]]

In [ ]:
# ✅ Check
check("1. X has 50 rows of 2", lambda: len(X) == 50 and all(len(row) == 2 for row in X) and X[0] == [x[0][0], x[1][0]])
check("2. transpose of a small matrix", lambda: transpose([[1, 2, 3], [4, 5, 6]]) == [[1, 4], [2, 5], [3, 6]])
check("2. the transpose of X is x", lambda: transpose(X) == x)

Two more helper functions from LELA60331 do the arithmetic: `dot(a, b)` (Week 4) and `mat_vec(matrix, vector)` (Week 7, Activity 10). Run the cell; it also defines the `sigmoid` function from LELA60331 Week 5.

In [ ]:
def dot(a, b):
    total = 0
    for i in range(len(a)):
        total += a[i] * b[i]
    return total

def mat_vec(matrix, vector):
    # the dot product of each row of the matrix with the vector
    return [dot(row, vector) for row in matrix]

def sigmoid(z):
    return 1 / (1 + math.exp(-z))

---
# Part 3 · Parameters and modules

## Parameters

Training changes the weights and the bias. For each of them we need to keep two things together: its current **value**, and its **gradient** (which way, and how much, to change it). A `Parameter` object holds both:

* `p.value`: a list of numbers (like a bias, one per output) or a matrix (like the weights)
* `p.grad`: a list or matrix of the **same shape**, which starts full of 0s

`zeros_like(value)` makes the 0s. It has to work for both shapes: `isinstance(value[0], list)` is `True` if the first element of `value` is itself a list, that is, if `value` is a matrix.

### 🏋️ Exercise 3 · Practice — `zeros_like` and `Parameter`

1. Complete `zeros_like(value)`: for a matrix, a matrix of 0s with the same number of rows and columns; for a list, a list of 0s of the same length. *Hint:* `[0] * 3` is `[0, 0, 0]`.
2. Complete the constructor of `Parameter`, which stores `value`, and a gradient of 0s with the same shape.

In [ ]:
def zeros_like(value):
    if isinstance(value[0], list):          # a matrix: a list of rows
        return ???      # 1.
    else:                                   # a list of numbers
        return ???

class Parameter:
    """Numbers that training changes: their values, and their gradients."""
    def __init__(self, value):
        self.value = ???                           # 2.
        self.grad = ???

p = Parameter([[1.5, -2.0, 0.5], [3.0, 1.0, 0.0]])
print(p.value)
print(p.grad)                      # you should get [[0, 0, 0], [0, 0, 0]]
print(Parameter([0.1, 0.2]).grad)  # you should get [0, 0]

In [ ]:
# ✅ Check
check("1. zeros_like for a matrix", lambda: zeros_like([[1, 2, 3], [4, 5, 6]]) == [[0, 0, 0], [0, 0, 0]])
check("1. zeros_like for a list", lambda: zeros_like([7, 8]) == [0, 0])
check("1. a new matrix, with separate rows", lambda: (lambda z: (z[0].append(1), z[1] == [0, 0, 0])[1])(zeros_like([[1, 2, 3], [4, 5, 6]])))
check("2. Parameter", lambda: p.value == [[1.5, -2.0, 0.5], [3.0, 1.0, 0.0]] and p.grad == [[0, 0, 0], [0, 0, 0]])

## Modules

Every layer, and every model, will be a subclass of a class called `Module` (Week 7, Part 4: inheritance). `Module` gives all of them two useful things for free. It is done for you: read it, then run the cell.

* **`__call__`**: a special method (Week 7, Part 3). Writing `layer(X)` calls `layer.__call__(X)`, which calls `layer.forward(X)`. So every module can be used like a function, and each subclass only has to write its own `forward` method.
* **`parameters()`**: returns a list of all the module's `Parameter` objects, **including those inside any modules it contains**. `vars(obj)` gives a dict of an object's instance variables, so the loop looks at each of them in turn: a `Parameter` is added to the list; a `Module` (for example, a layer inside a model) is asked for *its* parameters, and they are added with `extend`.

In [ ]:
class Module:
    """The parent class of every layer and every model."""
    def __init__(self):
        pass

    def __call__(self, X):
        return self.forward(X)

    def parameters(self):
        params = []
        for value in vars(self).values():
            if isinstance(value, Parameter):
                params.append(value)
            elif isinstance(value, Module):
                params.extend(value.parameters())
        return params


class Doubler(Module):
    """A tiny example module: it doubles every number, and has one (unused) parameter."""
    def __init__(self):
        super().__init__()
        self.scale = Parameter([2])
        self.name = "doubler"           # not a Parameter, so parameters() ignores it

    def forward(self, X):
        return [[2 * v for v in row] for row in X]

d = Doubler()
print(d([[1, 2], [3, 4]]))              # __call__ runs forward
print(vars(d))
print(len(d.parameters()), d.parameters()[0].value)

---
# Part 4 · A linear layer

A `Linear` layer does the matrix-vector multiplication from LELA60331 Week 7. It has `n_out` **outputs**, and for each example it gives one score per output. Logistic regression needs just one output (the score that the text is positive); later you will need more.

* `self.weight` is a `Parameter` holding a matrix with **one row per output** and one column per input feature, so `mat_vec(self.weight.value, x)` gives the scores (without the biases) of example `x`.
* `self.bias` is a `Parameter` holding one bias per output.
* Both start at random numbers between 0 and 1, as in LELA60331 Week 7.

**`forward(X)`** returns a matrix with one row per example and one score per output. It also keeps `X` in `self._X`, because the backward pass needs it.

**`backward(grad_out)`** is the **backward pass**. `grad_out` has the same shape as the output of `forward`: `grad_out[k][c]` says how much the loss changes when the score of output `c` for example `k` goes up. From it, `backward` **adds** each example's contribution to the gradients of the weights and the biases, just as in LELA60331 Week 7:

* the gradient for `weight[c][j]` gets `grad_out[k][c] * X[k][j]` from example `k` (the error times the feature)
* the gradient for `bias[c]` gets `grad_out[k][c]` from example `k`

(The last two lines return the gradients for the layer's **inputs**, which are needed when one layer feeds into another. They are done for you.)

### 🏋️ Exercise 4 · Practice — Complete `Linear`

Fill in the blanks 1–4.

In [ ]:
class Linear(Module):
    """A layer that gives n_out scores for each example: one row of weights and one bias per output."""
    def __init__(self, n_in, n_out):
        super().__init__()
        self.weight = Parameter([[random.random() for j in range(n_in)] for c in range(n_out)])
        self.bias = Parameter([random.random() for c in range(n_out)])

    def forward(self, X):
        self._X = X                                     # kept for the backward pass
        out = []
        for x in X:
            z = ???         # 1. one score per output
            for c in range(len(z)):
                z[c] += ???            # 2. plus that output's bias
            out.append(z)
        return out

    def backward(self, grad_out):
        W = self.weight.value
        for k in range(len(self._X)):                   # each example
            for c in range(len(W)):                     # each output
                for j in range(len(W[c])):              # each input feature
                    self.weight.grad[c][j] += ???    # 3.
                self.bias.grad[c] += ???                           # 4.
        # the gradients for the inputs (done for you)
        WT = transpose(W)
        return [mat_vec(WT, grad_out[k]) for k in range(len(self._X))]


layer = Linear(2, 3)
layer.weight.value = [[1, 0], [0, 1], [1, 1]]       # set the weights and biases by hand, to test it
layer.bias.value = [0, 10, 100]
print(layer([[2, 3], [1, 1]]))                       # you should get [[2, 13, 105], [1, 11, 102]]
layer.backward([[1, 0, 0], [0, 0, 2]])
print(layer.weight.grad, layer.bias.grad)            # you should get [[2, 3], [0, 0], [2, 2]] [1, 0, 2]

In [ ]:
# ✅ Check
def _layer():
    l = Linear(2, 3)
    l.weight.value = [[1, 0], [0, 1], [1, 1]]
    l.bias.value = [0, 10, 100]
    return l
check("the shapes", lambda: len(Linear(4, 2).weight.value) == 2 and len(Linear(4, 2).weight.value[0]) == 4 and len(Linear(4, 2).bias.value) == 2)
check("1–2. forward", lambda: _layer()([[2, 3], [1, 1]]) == [[2, 13, 105], [1, 11, 102]])
def _back():
    l = _layer()
    l([[2, 3], [1, 1]])
    l.backward([[1, 0, 0], [0, 0, 2]])
    return l.weight.grad == [[2, 3], [0, 0], [2, 2]] and l.bias.grad == [1, 0, 2]
check("3–4. backward", _back)
check("a Linear layer has two parameters", lambda: [len(p.value) for p in Linear(2, 3).parameters()] == [3, 3])

🔮 **Predict:** what happens to the gradients if you call `layer.backward(...)` **twice** without resetting them? Why might that be a problem in a training loop? (Part 5 deals with it.)

---
# Part 5 · A loss and an optimiser

## The loss

The loss measures how wrong the model's scores are. `BCEWithLogitsLoss` ("binary cross-entropy, from the scores") is the loss from LELA60331 Week 5, for two classes: it turns each score into a probability with `sigmoid`, then takes the average of minus the log of the probability given to the correct label.

Called as `loss_fn(scores, y)`, it returns the average loss, and also keeps the probabilities and the labels. Its method `grad()` then starts the backward pass: it returns, for each example, how much the loss changes when that example's score goes up. For this loss that is exactly the **error** from LELA60331 Week 5, divided by the number of examples (because the loss is an average):

`(q[k] - y[k]) / n`

It returns these as a matrix with one row per example and one number in each row, the same shape as the scores, so that it can be passed straight to `backward`.

## The optimiser

`SGD` ("stochastic gradient descent") updates the parameters. It is given the list of parameters when it is made, and has two methods:

* `zero_grad()` sets every parameter's gradient back to 0s. Because `backward` **adds** to the gradients, we must reset them before each backward pass.
* `step()` updates every parameter: **new value = old value − learning rate × gradient**. A parameter's value can be a matrix or a list, so it needs two cases, as in `zeros_like`.

### 🏋️ Exercise 5 · Practice — `BCEWithLogitsLoss` and `SGD`

Fill in the blanks 1–5.

In [ ]:
class BCEWithLogitsLoss:
    """The cross-entropy loss for two classes, from the scores."""
    def __call__(self, scores, y):
        self._q = ???                  # 1. the probabilities
        self._y = y
        eps = 0.00001          # a tiny number, so that we never take the log of 0
        losses = []
        for k in range(len(y)):
            if y[k] == 1:
                losses.append(-math.log2(self._q[k] + eps))
            else:
                losses.append(-math.log2(1 - self._q[k] + eps))
        return sum(losses) / len(losses)

    def grad(self):
        n = len(self._y)
        return ???     # 2.


class SGD:
    """Gradient descent: new value = old value - learning rate * gradient."""
    def __init__(self, parameters, lr):
        self.parameters = parameters
        self.lr = lr

    def zero_grad(self):
        for p in self.parameters:
            p.grad = ???                                 # 3.

    def step(self):
        for p in self.parameters:
            if isinstance(p.value[0], list):        # a matrix
                for i in range(len(p.value)):
                    for j in range(len(p.value[i])):
                        p.value[i][j] -= ???          # 4.
            else:                                   # a list
                for i in range(len(p.value)):
                    p.value[i] -= ???                    # 5.


loss_fn = BCEWithLogitsLoss()
print(round(loss_fn([[0.0], [2.0]], [1, 0]), 3))          # you should get 2.034
print([[round(v, 3) for v in row] for row in loss_fn.grad()])     # you should get [[-0.25], [0.44]]

In [ ]:
# ✅ Check
check("1. the loss", lambda: round(BCEWithLogitsLoss()([[0.0], [2.0]], [1, 0]), 3) == 2.034)
def _grad():
    f = BCEWithLogitsLoss()
    f([[0.0], [2.0]], [1, 0])
    return [[round(v, 3) for v in row] for row in f.grad()] == [[-0.25], [0.44]]
check("2. grad", _grad)
def _sgd():
    w = Parameter([[1.0, 2.0], [3.0, 4.0]])
    b = Parameter([1.0, 1.0])
    w.grad = [[10, 0], [0, -10]]
    b.grad = [1, 2]
    opt = SGD([w, b], 0.1)
    opt.step()
    ok = w.value == [[0.0, 2.0], [3.0, 5.0]] and b.value == [0.9, 0.8]
    opt.zero_grad()
    return ok and w.grad == [[0, 0], [0, 0]] and b.grad == [0, 0]
check("3–5. SGD", _sgd)

---
# Part 6 · The model and the training loop

Now we put the parts together. A logistic regression model is a `Module` that **has** one `Linear` layer, with one input per feature and **one** output (the score):

* `forward(X)` passes `X` through the layer and returns the scores
* `backward(grad)` passes the gradients back through the layer

Because the layer is stored in an instance variable, `model.parameters()` (inherited from `Module`) finds the layer's weights and bias automatically.

### 🏋️ Exercise 6 · Practice — `LogisticRegression` and the training loop

1. Complete `LogisticRegression`.
2. Complete the training loop. Each pass through the data (each **epoch**):
   * forgets the old gradients
   * calculates the scores (the **forward pass**)
   * calculates the loss, and records it
   * calculates the gradients (the **backward pass**): `model.backward(loss_fn.grad())`
   * updates the parameters

The settings are those of LELA60331 Week 7: a learning rate of 0.5 and 1000 epochs, with `random.seed(10)` before the model is made.

In [ ]:
class LogisticRegression(Module):
    """Logistic regression: a model with one Linear layer and one output."""
    def __init__(self, n_features):
        ???                          # 1.
        self.linear = ???

    def forward(self, X):
        return ???

    def backward(self, grad):
        return ???


random.seed(10)
model = LogisticRegression(2)
loss_fn = BCEWithLogitsLoss()
optimizer = SGD(model.parameters(), lr=0.5)
n_epochs = 1000
loss_history = []

for epoch in range(n_epochs):                         # 2.
    ???
    scores = ???
    loss = ???
    loss_history.append(loss)
    ???
    ???

plt.plot(loss_history)
plt.xlabel("epoch")
plt.ylabel("loss")
plt.show()
print(model.linear.weight.value, model.linear.bias.value)      # about [[1.17, -1.32]] and [0.51]
print("final loss:", round(loss_history[-1], 3))              # you should get 0.656

In [ ]:
# ✅ Check
check("1. the model has 2 parameters: the weights and the bias", lambda: len(LogisticRegression(2).parameters()) == 2)
check("1. one output", lambda: len(LogisticRegression(5).linear.weight.value) == 1 and len(LogisticRegression(5).linear.weight.value[0]) == 5)
check("2. the loss went down", lambda: loss_history[-1] < loss_history[0])
check("2. the same weights as LELA60331 Week 7", lambda: [round(w, 2) for w in model.linear.weight.value[0]] == [1.17, -1.32] and round(model.linear.bias.value[0], 2) == 0.51)
check("2. final loss about 0.656", lambda: abs(loss_history[-1] - 0.656) < 0.001)

These are the weights from LELA60331 Week 7. Compare the two pieces of code: which lines in the seminar's training loop does each line of this loop replace?

---
# Part 7 · Using and evaluating the model

The model gives **scores**. To use it, we write three small functions that work with **any** model with one output, so they are kept outside the model classes:

* `predict_proba(model, X)`: the probability that each example is positive: the sigmoid of its score
* `predict(model, X)`: the predicted class of each example: 1 if its probability is above 0.5, and 0 otherwise
* `evaluate(model, X, y)`: a **dict** (Week 2) with the accuracy, precision and recall (LELA60331 Week 7, Activities 2 and 4). The counting is done for you.

### 🏋️ Exercise 7 · Practice — Utilities

1. Complete the three functions.
2. `results`: the trained model's results on `X` and `y`
3. `new_probs`: the probability that each of these two new texts is positive (LELA60331 Week 7, Activity 3). Remember that the features are **log** counts, and that a model always takes a **list** of examples.
   * a) 2 positive words and 3 negative words
   * b) 10 positive words and 1 negative word

In [ ]:
def predict_proba(model, X):
    return ???                 # 1.

def predict(model, X):
    return ???

def evaluate(model, X, y):
    predicted = ???
    correct = 0
    tp = 0
    fp = 0
    fn = 0
    for k in range(len(y)):
        if predicted[k] == y[k]:
            correct += 1
        if predicted[k] == 1 and y[k] == 1:
            tp += 1
        if predicted[k] == 1 and y[k] == 0:
            fp += 1
        if predicted[k] == 0 and y[k] == 1:
            fn += 1
    return {"accuracy": ???, "precision": ???, "recall": ???}

results = ???                                    # 2.
new_probs = ???    # 3.
print(results)
print(new_probs)                 # about 0.466 and 0.961

In [ ]:
# ✅ Check
check("1. predict_proba", lambda: abs(predict_proba(model, [[0, 0]])[0] - sigmoid(model.linear.bias.value[0])) < 1e-12)
check("1. predict", lambda: predict(model, X[:5]) == [int(p > 0.5) for p in predict_proba(model, X[:5])])
check("2. accuracy 0.78, precision 0.75, recall 0.783",
      lambda: results["accuracy"] == 0.78 and results["precision"] == 0.75 and abs(results["recall"] - 0.783) < 0.001)
check("3. new_probs", lambda: abs(new_probs[0] - 0.466) < 0.005 and abs(new_probs[1] - 0.961) < 0.005)

### 🏋️ Exercise 8 · Practice — Several models side by side

In a notebook written as a series of cells, trying a different learning rate means changing `lr` and re-running the training cell, which replaces the old model. With classes, each model is a separate object, so you can keep several and compare them.

1. Complete `train(model, X, y, lr, n_epochs)`, which trains `model` with the loop from Exercise 6 and returns its loss history. (It makes its own loss and optimiser.)
2. `models`: a dict from each learning rate in `rates` to a new `LogisticRegression`, trained with that learning rate
3. `histories`: a dict from each learning rate to the loss history that `train` returned

The cell then plots the three loss histories on the same axes. 🔮 **Predict** before you run it: what will each curve look like? (Think about what you saw when you changed the learning rate for linear regression in LELA60331 Week 4.)

In [ ]:
def train(model, X, y, lr, n_epochs):
    loss_fn = BCEWithLogitsLoss()
    optimizer = ???                       # 1.
    history = []
    for epoch in range(n_epochs):
        optimizer.zero_grad()
        loss = ???
        history.append(loss)
        ???
        ???
    return history

rates = [0.01, 0.5, 2]
models = {}
histories = {}
for lr in rates:
    random.seed(10)
    models[lr] = ???                               # 2.
    histories[lr] = ???                # 3.

for lr in rates:
    plt.plot(histories[lr], label="lr = " + str(lr))
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.show()
for lr in rates:
    print(lr, round(histories[lr][-1], 3), evaluate(models[lr], X, y))

In [ ]:
# ✅ Check
check("1–2. three models", lambda: sorted(models) == rates and all(isinstance(m, LogisticRegression) for m in models.values()))
check("3. each trained for 1000 epochs", lambda: all(len(h) == 1000 for h in histories.values()))
check("lr = 0.5 gives the same as Exercise 6", lambda: abs(histories[0.5][-1] - loss_history[-1]) < 1e-9)
check("the first model is unchanged", lambda: abs(loss_history[-1] - 0.656) < 0.001)

⭐ **Challenge (optional):** add a method `__str__` to `LogisticRegression`, so that `print(model)` shows its weights and bias, rounded to 2 decimal places.

---
## ✏️ Summary

You can now:

* explain why a program with many shared global names is fragile, and how classes help
* split a model into small parts with one job each: **parameters**, a **layer**, a **loss** and an **optimiser**
* use a parent class (`Module`) to give every layer and model `__call__` and `parameters()`
* write a `Linear` layer with a **forward** pass and a **backward** pass
* train a model with a loop of five lines: `zero_grad`, forward, loss, backward, `step`
* keep several models side by side, and evaluate them with functions that work for any model